In [9]:
import os
from typing import Annotated, Any, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph
from langchain.chat_models import init_chat_model

In [3]:
class AgentState(TypedDict):
    """State of the agent."""
    messages: list[BaseMessage]

In [4]:
@tool
def calculator(expression: str) -> str:
    """Avalia uma expressão matemática simples."""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Erro na expressão"
    
@tool
def web_search(query: str) -> str:
    """Simula uma busca na web."""
    return f"Resultados da busca para '{query}'"

@tool
def get_weather(location: str) -> str:
    """Simula a obtenção de informações meteorológicas."""
    return f"Previsão do tempo para {location}: Ensolarado, 25°C"

In [5]:
tools = [calculator, web_search, get_weather]

In [10]:
llm = init_chat_model("openai:gpt-4.1-nano")

In [11]:
llm.invoke("Olá, quais ferramentas você tem disponível?")

AIMessage(content='Olá! Eu sou uma inteligência artificial treinada para ajudar com uma variedade de tarefas, como responder perguntas, fornecer informações, ajudar na redação de textos, oferecer orientações sobre diversos assuntos, entre outras coisas. No momento, não tenho acesso a ferramentas externas em tempo real, mas posso ajudar com conhecimentos, geração de conteúdo e suporte em várias áreas. Como posso ajudar você hoje?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 15, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7d0ce1a5ac', 'id': 'chatcmpl-Dci1afSyigS64PtskOLEzalxal1ly', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': N

In [12]:
llm_with_tools = llm.bind_tools(tools)

In [13]:
llm_with_tools.invoke("Olá, quais ferramentas você tem disponível?")

AIMessage(content='Olá! Tenho disponível as seguintes ferramentas:\n\n1. **functions.calculator**: Para avaliar expressões matemáticas simples.\n2. **functions.web_search**: Para realizar buscas na web.\n3. **functions.get_weather**: Para obter informações meteorológicas de uma localização específica.\n\nSe precisar de alguma delas, é só pedir!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 96, 'total_tokens': 163, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_bfd5140507', 'id': 'chatcmpl-Dci2E5Io6g0bcJDZpLua4kxcL2bVk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0019-1c0c-78a0-b760-0b5180f87121-0', usage_metadata={'input_tok

In [22]:
response = llm_with_tools.invoke("qual a previsão do tempo para São Paulo e qual o placar do jogo do atletico?")

In [23]:
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'São Paulo'}, 'id': 'call_4ap228byRMdeF7ai6mh03KvZ', 'type': 'tool_call'}, {'name': 'web_search', 'args': {'query': 'resultado do jogo do Atlético'}, 'id': 'call_2POeLeyXIvGREX1wsdbRL7zP', 'type': 'tool_call'}]


In [27]:
tool_map = {tool.name: tool for tool in tools}

In [28]:
tool_map['get_weather']

StructuredTool(name='get_weather', description='Simula a obtenção de informações meteorológicas.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001D37D273060>)

In [32]:
if response.tool_calls:
    for tool_call in response.tool_calls:
        tool = tool_map[tool_call.get("name")]
        tool_args = tool_call.get("args", {})
        print(tool_args)

{'location': 'São Paulo'}
{'query': 'resultado do jogo do Atlético'}
